# 개별종목 조합G — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합G 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합G의 피처 값만 지정합니다.
import json

COMBINATION = 'G'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_20',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합G 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_20', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4563,0.5012,-0.0449,0.3504,0.3621,0.0598,0.3693,0.1757,0.2794
1,2,balanced,980,20150123,20150421,0.3824,0.3978,-0.0154,0.3574,0.3627,0.0518,0.3647,0.2558,0.3218
2,3,balanced,1210,20151228,20160328,0.3573,0.3762,-0.0188,0.3503,0.3509,0.0306,0.3614,0.2859,0.3278
3,4,balanced,1439,20161202,20170228,0.4568,0.4617,-0.0049,0.3940,0.4001,0.1155,0.4153,0.1934,0.3031
4,5,balanced,1669,20171113,20180207,0.3993,0.3901,0.0092,0.3833,0.3856,0.0830,0.3891,0.3453,0.3745
5,6,balanced,1899,20181024,20190118,0.4067,0.3725,0.0342,0.4049,0.4055,0.1110,0.4305,0.3765,0.3956
6,7,balanced,2129,20190930,20191224,0.4399,0.4781,-0.0383,0.3654,0.3747,0.0791,0.4065,0.2587,0.3380
7,8,balanced,2359,20200902,20201130,0.3748,0.3476,0.0272,0.3712,0.3808,0.0713,0.3873,0.4715,0.4009
8,9,balanced,2589,20210806,20211105,0.3544,0.3914,-0.0370,0.3499,0.3585,0.0319,0.3675,0.3140,0.3384
9,10,balanced,2818,20220714,20221012,0.3865,0.3454,0.0411,0.3854,0.3944,0.0898,0.3884,0.2942,0.3496


,OOS 폴드 평균
accuracy,0.4000
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0031
macro_f1,0.3743
balanced_accuracy,0.3796
mcc,0.0748
pr_auc_macro_ovr,0.3891
down_recall,0.3183
core_harmonic_mean,0.3527


재실행 명령: python scripts/run_stock_model_experiment.py
